# Adversarial Attack Benchmark Pipeline
This notebook runs the full pipeline for evaluating HSI models (HybridSN, S3ANet, SACNet, SpectralFormer) against adversarial attacks (FGSM, I-FGSM, PGD, SS-FGSM, RTAA).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/S3Anet_data'
REPO_DIR = '/content/RTAA'

## 1. Setup Environment and Repositories

In [ ]:
!git clone https://github.com/YichuXu/S3ANet.git /content/S3ANet
!git clone https://github.com/YonghaoXu/SACNet.git /content/SACNet
!git clone https://github.com/danfenghong/IEEE_TGRS_SpectralFormer.git /content/SpectralFormer

!mkdir -p /content/S3ANet/Data/PaviaU /content/S3ANet/Data/IndianPines /content/S3ANet/Data/Salinas
!mkdir -p /content/SACNet/Data/PaviaU /content/SACNet/Data/IndianPines /content/SACNet/Data/Salinas

# Link data into S3ANet and SACNet Data folders (Assuming S3Anet_data has the .mat files)
!ln -s /content/drive/MyDrive/S3Anet_data/* /content/S3ANet/Data/PaviaU/
!ln -s /content/drive/MyDrive/S3Anet_data/* /content/SACNet/Data/PaviaU/
!ln -s /content/drive/MyDrive/S3Anet_data/* /content/S3ANet/Data/IndianPines/
!ln -s /content/drive/MyDrive/S3Anet_data/* /content/SACNet/Data/IndianPines/
!ln -s /content/drive/MyDrive/S3Anet_data/* /content/S3ANet/Data/Salinas/
!ln -s /content/drive/MyDrive/S3Anet_data/* /content/SACNet/Data/Salinas/

!pip install scikit-image openpyxl

## 2. Generate Dataset Splits (S3ANet & SACNet)

In [ ]:
# Note: GenSample.py arguments might need adjustment based on the original repositories.
!cd /content/S3ANet && python GenSample.py --dataID 1 --train_samples 300
!cd /content/SACNet && python GenSample.py --dataID 1 --train_samples 300

## 3. Train Classifiers

In [ ]:
import os
os.environ['PYTHONPATH'] = f"/content/RTAA/src:{os.environ.get('PYTHONPATH', '')}"

# Train HybridSN
!python /content/RTAA/RTAA_S/scripts/train_classifier.py --dataset PaviaU
!python /content/RTAA/RTAA_S/scripts/train_classifier.py --dataset IndianPines
!python /content/RTAA/RTAA_S/scripts/train_classifier.py --dataset Salinas

# Train S3ANet
!python /content/RTAA/RTAA_S/scripts/train_s3anet.py --dataset PaviaU --classes 9 --bands 103
!python /content/RTAA/RTAA_S/scripts/train_s3anet.py --dataset IndianPines --classes 16 --bands 200
!python /content/RTAA/RTAA_S/scripts/train_s3anet.py --dataset Salinas --classes 16 --bands 204

# Train SACNet
!python /content/RTAA/RTAA_S/scripts/train_sacnet.py --dataset PaviaU --classes 9 --bands 103
!python /content/RTAA/RTAA_S/scripts/train_sacnet.py --dataset IndianPines --classes 16 --bands 200
!python /content/RTAA/RTAA_S/scripts/train_sacnet.py --dataset Salinas --classes 16 --bands 204


## 4. Train RTM Surrogates

In [ ]:
# Generates surrogates for 103, 200, 204 bands for the 3 datasets
!python /content/RTAA/src/rtaa/rtm/train_surrogate.py --n-bands 103 --out checkpoints/rtm_surrogate_103bands.pt
!python /content/RTAA/src/rtaa/rtm/train_surrogate.py --n-bands 200 --out checkpoints/rtm_surrogate_200bands.pt
!python /content/RTAA/src/rtaa/rtm/train_surrogate.py --n-bands 204 --out checkpoints/rtm_surrogate_204bands.pt


## 5. Run Benchmark and Generate Excel

In [ ]:
!python /content/RTAA/RTAA_S/scripts/run_full_benchmark.py --data-dir {DATA_DIR} --out /content/drive/MyDrive/benchmark_results.xlsx
print("Benchmark completed! Excel file saved to Google Drive.")